In [ ]:
# RAG : 검색해서 답변하는 전체 방식
# LangChain : 그 방식을 조립하기 쉽게 해 주는 도구
# LangGraph : 그 방식을 그래프와 엣지를 이용해 조립하기 쉽게 해 주는 도구

!pip install openai sentence-transformers chromadb python-dotenv
!pip install -U langchain langchain-community
!pip install langchain-text-splitters


In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-4o-mini"
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
# RAG 첫단계 : text를 읽어 vectordb에 저장
# 방법1 : 문서 데이터 직접 작성
# documents = [
#     "김치찌게는 한국의 대표적인 찌게 요리이다",
#     "된장찌게는 발효된 된장을 이용해 만든다",
#     "비빕밥은 여러 가지 나물을 비벼서 먹는 밥 요리이다",
#     "불고기는 양념한 소고기를 구워 먹는 전통 음식이다",
#     "삼계탕은 닭에 인삼, 대추 등을 넣고 푹 끓인 보양식이다"
# ]

# 방법2 : 문서 파일 읽기 (python)
# with open('foods.txt', 'r', encoding='utf-8') as f:
#     documents = [line.strip() for line in f if line.strip()]
# print(documents)

# 방법3 : 문서 파일 읽기 (LangChain)
from langchain_community.document_loaders import TextLoader
# loader = TextLoader('foods.txt', encoding='utf-8')
# datas = loader.load()
# documents = datas[0].page_content.split('\n')
# # documents 리스트에서 빈 문자열 제거 후, 앞뒤 공백을 정리
# documents = [doc.strip() for doc in documents if doc.strip()]
# print(documents)

# 방법4 : 문서 파일 읽기 (LangChain)
from langchain_text_splitters import CharacterTextSplitter
loader = TextLoader('foods.txt', encoding='utf-8')
datas = loader.load()
# text 나누기
splitter = CharacterTextSplitter(separator='\n', chunk_size=100, chunk_overlap=0)
spl_docs = splitter.split_documents(datas)  # Document 객체를 여러 조각으로 분할

# Document 객체에서 실제 문자열만 추출
documents = [doc.page_content for doc in spl_docs]
print(documents)
print(len(documents), documents[0])


In [ ]:
# 임베딩 벡터로 변환
doc_embeddings = embedder.encode(documents)
print(doc_embeddings[0][:3])

# ChromaDB에 저장
# 방법1 : 임시 저장
# chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# 방법2 : 영구 저장
# chroma_client = chromadb.PersistentClient(path="./chroma_db")  # 세부 설정 불가(기본값으로 설정)
chroma_client = chromadb.Client(Settings(
    persist_directory="./chroma_db",
    anonymized_telemetry=False    # 익명으로 자료 수집 불가(보안 강화 목적)
))   # 세부 설정 가능

collection = chroma_client.get_or_create_collection(name="foods")

# 기존 데이터 지우고 컬렉션 새로 작성
# chroma_client.delete_collection("foods")
# collection = chroma_client.create_collection(name="foods")

# ChromaDB에 저장
for i, (doc, embedding) in enumerate(zip(documents, doc_embeddings)):
    collection.add(
        documents=[doc],
        embeddings=[embedding.tolist()],
        ids=[f'doc_{i}']
    )



In [ ]:
# 다음 단계 : 질문 -> 관련문서 검색 -> LLM 응답 생성
# RAG (Retrieval 단계 - 관련문서 검색)
query = "한국의 대표적인 찌게 음식을 알려줘"    # 사용자의 기본 질문

# 질문과 유사한 문자열을 ChromaDB에서 일기

query_embedding = embedder.encode(
    [query],
    normalize_embeddings=True
)[0]
# print(query_embedding)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results = len(documents),
    include = ['documents', 'distances']
)

print('results : ', results)

# 질문에 대한 거리값 목록 출력
import numpy as np
distances = results['distances'][0]
print('거리(distance)값 : ', distances)

# 직접 계산한 유사도 값으로 확인
similarities = []
for doc_id in results["ids"][0]:
    doc_embed = collection.get(
        ids = [doc_id],
        include=['embeddings']
    )['embeddings'][0]

    doc_embed = np.array(doc_embed, dtype=float)  # 계산을 위해 numpy 배열로 변환
    sim = np.dot(query_embedding, doc_embed)
    similarities.append(sim)

print('유사도(similarity) 값 : ', similarities)

ids_list = results["ids"][0]
docs_list = results["documents"][0]
dists_list = results["distances"][0]

for i, doc_id in enumerate(ids_list):
    print(f'문서 {i}')
    print(f' - id:{doc_id}')
    print(f' - 문서 내용:\n{docs_list[i]}')
    print(f' - 거리(distance):\n{dists_list[i]:.4f}')
    print(f' - 유사도(similarity):\n{similarities[i]:.4f}')
    print(f'-------------')


In [ ]:
# RAG (generation 단계 - 검색된 문서로 질문을 보강해서 LLM에서 답변 요청 후 생성)
retrieved_docs_list = results['documents'][0]  # ChromaDB에서 검색된 문서 리스트를 꺼냄
retrieved_docs = "\n\n".join(retrieved_docs_list)
# print(retrieved_docs)

prompt = f"""
너는 한국 전통 음식 전문가야.
지금부터 사용자 질문에 답변 할 때는 반드시 아래 문서 내용을 참고해.

[참고문서]
{retrieved_docs}

[사용자 질문]
{query}

위 참고 문서를 바탕으로 사용자 질문에 10문장 이내로 답해 줘
마크다운 기호없이 평문으로 답해 줘
"""
print(prompt)


In [ ]:
# LLM 응답 - OpenAI 서버에게 요청
response = client.responses.create(
    model = model,
    input = prompt
)

import re
text = response.output_text
# print(text)
sentences = re.split(r'(?<=[.!? ])\s+', text)  # 마침표, 물음표, 느낌표 뒤에서 문장 문리
for s in sentences:
    s = s.strip()
    if not s:
        continue

    print(s)